In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.1"

import numpy as np
import matplotlib.pyplot as plt
import cluster as cl
import jax
import fmdj
import aegis
import json

In [ ]:
prof = aegis.profiles.NFWProfile(6., m200c=1e14)
print(prof.rs, prof.r200c)
prof.tcirc(prof.r200c, inyears=True) / 1e9

In [ ]:
file = "logs/run_8.npz"
res = np.load(file)
print(res["history"]["parameters"].dtype.names)

In [ ]:
def closest(x1, x2):
    r2mat = np.sum((x1[...,:,None,:] - x2[...,None,:,:])**2, axis=-1)
    return np.sqrt(np.min(r2mat, axis=-1))
def l10mtot(l10m):
    return np.log10(np.sum(10.**l10m, axis=-1))

def plot_history(file):
    res = np.load(file)
    history = res["history"]
    parameters = history["parameters"]
    target = res["target_parameters"]
    log10_mass = np.log10(np.asarray(cl.mass_from_exp.jit(parameters["exp_m"])))
    target_log10_mass = np.log10(np.asarray(cl.mass_from_exp.jit(target["exp_m"])))
    scale_radius = np.asarray(cl.radius_from_exp.jit(parameters["exp_rs"]))
    target_scale_radius = np.asarray(cl.radius_from_exp.jit(target["exp_rs"]))

    fig, axs = plt.subplots(5,1, figsize=(6,10), sharex=True)
    fig.subplots_adjust(hspace=0.02)
    axs[0].semilogy(history["step"], history["loss"])
    axs[0].axhline(res["optimal_loss"], ls="dotted", color="black")

    axs[1].plot(history["step"], log10_mass)
    axs[1].plot(history["step"], l10mtot(log10_mass), color="black")
    for v in target_log10_mass:
        axs[1].axhline(v, color="grey", ls="dotted")
    axs[1].axhline(l10mtot(target_log10_mass), color="black", ls="dotted")

    axs[2].plot(history["step"], np.log10(scale_radius*1e3))
    for v in target_scale_radius:
        axs[2].axhline(np.log10(v*1e3), color="grey", ls="dotted")
    
    rclosest = closest(parameters["position"], target["position"])
    axs[3].semilogy(history["step"], rclosest*1e3)

    vclosest = closest(parameters["velocity"], target["velocity"])
    axs[4].semilogy(history["step"], vclosest*1e3)


    axs[0].set_ylabel("loss")
    axs[1].set_ylabel(r"$\log_{10} (M/M_{\odot})$")
    axs[2].set_ylabel(r"$\log_{10} (r_s/\mathrm{kpc})$")
    axs[3].set_ylabel(r"$r_{\mathrm{closest}}$ [kpc]")
    axs[4].set_ylabel(r"$v_{\mathrm{closest}}$ [km/s]")

    axs[1].set_ylim(11.5,13.5)

In [ ]:
plot_history("logs/run_1.npz")

In [ ]:
res = np.load("logs/run_1.npz")
cfg = cl.Config.from_json("logs/run_1.json")
par = res["target_parameters"]

part_target = cl.get_particles(par, cfg, cfg.target_particle_seed)

In [ ]:
plt.scatter(part_target.pos[::100,0], part_target.pos[::100,1], marker=".", alpha=0.1)

